In [1]:
!git lfs install


Git LFS initialized.


In [2]:
!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B


Cloning into 'Qwen2.5-0.5B'...
Updating files:  80% (8/10)
Updating files:  90% (9/10)
Updating files: 100% (10/10)
Updating files: 100% (10/10), done.


In [3]:
!ls -lh Qwen2.5-0.5Bs


'ls' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import os
p = "Qwen2.5-0.5B"
for f in sorted(os.listdir(p)):
    fp = os.path.join(p, f)
    if os.path.isfile(fp):
        print(f"{f:<30} {os.path.getsize(fp)/1e6:>10.2f} MB")

.gitattributes                       0.00 MB
LICENSE                              0.01 MB
README.md                            0.00 MB
config.json                          0.00 MB
generation_config.json               0.00 MB
merges.txt                           1.82 MB
model.safetensors                  988.10 MB
tokenizer.json                       7.33 MB
tokenizer_config.json                0.01 MB
vocab.json                           2.78 MB


In [5]:
%cd Qwen2.5-0.5B
!git lfs pull
%cd ..

c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 3\1 dars\Qwen2.5-0.5B
c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 3\1 dars


In [7]:

%pip install "huggingface-hub>=1.5.0,<2.0"


  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)

  Attempting uninstall: h11

    Found existing installation: h11 0.16.0

    Uninstalling h11-0.16.0:

      Successfully uninstalled h11-0.16.0

   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
   ---------------------------------------- 0/2 [h11]
  Attempting uninstall: huggingface-hub
   ---------------------------------------- 0/2 [h11]
    Found existing installation: huggingface_hub 2.2.0
   ------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore2 2.13.1 requires h11>=0.16, but you have h11 0.14.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\xolmu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

cuda = torch.cuda.is_available()
dtype = torch.float16 if cuda else torch.float32   # CPU'da float16 ishlamaydi

tok = AutoTokenizer.from_pretrained("Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen2.5-0.5B", torch_dtype=dtype).eval()
if cuda:
    model.cuda()

tok.pad_token = tok.pad_token or tok.eos_token
print("Qurilma:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qurilma: cpu


In [9]:
SAVOLLAR = [
    "Google nima?",
    "Italiya poytaxti qayer?",
    "Menga AIni organish uchun 3 ta maslahat ber.",
    "havo sovuq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 5 nechaga teng?",
]

def javob(savol):
    inp = tok(savol, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=100, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True).strip()

natijalar = [{"savol": s, "base_javob": javob(s)} for s in SAVOLLAR]

for r in natijalar:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["base_javob"])
    print("-" * 70)

json.dump(natijalar, open("baseline_1dars.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saqlandi: baseline_1dars.json")

SAVOL: Google nima?
JAVOB: - Google News
Google nima?
O que é o Google NIMA?
A partir de 2013, a empresa lançou um novo serviço para os usuários do Google. O Google NIMA é uma ferramenta de busca inteligente que permite aos usuários pesquisar em busca de informações mais precisas e eficientes.
O Google NIMA é um serviço de pesquisa inteligente que utiliza técnicas de aprendizado automático para ajudar os usuários a encontrar informações
----------------------------------------------------------------------
SAVOL: Italiya poytaxti qayer?
JAVOB: - 10.05.2019 | 13:47
Poytaxti qayer, kahin bokhsharq, khalifat al-islam, khalifat al-muqtada, khalifat al-mujahid, khalifat al-muslimek, khalifat al-murabit, khalifat al-masih, khalifat al-m
----------------------------------------------------------------------
SAVOL: Menga AIni organish uchun 3 ta maslahat ber.
JAVOB: 1. Mengeksekusi peraturan pemerintah untuk memperkuat kesehatan masyarakat, yaitu dengan mengembangkan program kesehatan umum dan

In [10]:
natijalar = []
for i, s in enumerate(SAVOLLAR, 1):
    print(f"[{i}/5]...")
    natijalar.append({"savol": s, "base_javob": javob(s)})

[1/5]...
[2/5]...
[3/5]...
[4/5]...
[5/5]...


# Instruct modelda sinash

In [1]:
!git lfs install

Git LFS initialized.


In [2]:

!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

Cloning into 'Qwen2.5-0.5B-Instruct'...
Updating files:  80% (8/10)
Updating files:  90% (9/10)
Updating files: 100% (10/10)
Updating files: 100% (10/10), done.


In [3]:
!ls -lh Qwen2.5-0.5B-Instruct

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!du -sh Qwen2.5-0.5B-Instruct

'du' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
import os
p = "Qwen2.5-0.5B-Instruct"
for f in sorted(os.listdir(p)):
    fp = os.path.join(p, f)
    if os.path.isfile(fp):
        print(f"{f:<30} {os.path.getsize(fp)/1e6:>10.2f} MB")

.gitattributes                       0.00 MB
LICENSE                              0.01 MB
README.md                            0.01 MB
config.json                          0.00 MB
generation_config.json               0.00 MB
merges.txt                           1.82 MB
model.safetensors                  988.10 MB
tokenizer.json                       7.33 MB
tokenizer_config.json                0.01 MB
vocab.json                           2.78 MB


In [8]:
%cd Qwen2.5-0.5B-Instruct
!git lfs pull
%cd ..

c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 3\1 dars\Qwen2.5-0.5B-Instruct
c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 3\1 dars


In [9]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

cuda = torch.cuda.is_available()
dtype = torch.float16 if cuda else torch.float32   # CPU'da float16 ishlamaydi

tok = AutoTokenizer.from_pretrained("Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen2.5-0.5B-Instruct", torch_dtype=dtype).eval()
if cuda:
    model.cuda()

tok.pad_token = tok.pad_token or tok.eos_token
print("Qurilma:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qurilma: cpu


In [15]:
SAVOLLAR = [
    "Google nima?",
    "Italiya poytaxti qayer?",
    "Menga AIni organish uchun 3 ta maslahat ber.",
    "havo sovuq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 5 nechaga teng?",
]

def javob(savol):
    inp = tok(savol, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=100, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True).strip()

natijalar = [{"savol": s, "instruct_javob": javob(s)} for s in SAVOLLAR]

for r in natijalar:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["instruct_javob"])
    print("-" * 70)

json.dump(natijalar, open("instruct_model_1dars.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saqlandi: instruct_model_1dars.json")

SAVOL: Google nima?
JAVOB: - Wikipedia
Nim (pronounced "nimm") is a game of chance played on a 15x15 grid. The objective is to reach the goal by moving pieces from one cell to another, with each move being either forward or backward. Pieces can be moved in any direction, including diagonally.
The game was invented by the American mathematician John von Neumann and published in 1948 as part of his book "Mathematical Foundations of Computer Science".
----------------------------------------------------------------------
SAVOL: Italiya poytaxti qayer?
JAVOB: Qaydani, 2014

Translate to English

Italian: "Poytaxti qayer? Qaydani, 2014"

English translation:

"Did you know that?"

Qaydani, 2014

This appears to be a title or heading in Arabic. The phrase translates roughly to:

"Do you know that?"

or

"Did you learn that?"

The question is asking if someone has learned something new about the year
----------------------------------------------------------------------
SAVOL: Menga AIni orga

In [16]:
natijalar = []
for i, s in enumerate(SAVOLLAR, 1):
    print(f"[{i}/5]...")
    natijalar.append({"savol": s, "instruct_javob": javob(s)})

[1/5]...
[2/5]...
[3/5]...
[4/5]...
[5/5]...


In [17]:

import json
import torch

SAVOLLAR = [
    "Google nima?",
    "Italiya poytaxti qayer?",
    "Menga AIni organish uchun 3 ta maslahat ber.",
    "havo sovuq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 5 nechaga teng?",
]

model.eval()

def javob(savol):
    messages = [
        {"role": "user", "content": savol}
    ]

    # Chat template orqali tokenlarga aylantirish
    inp = tok.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Javob yaratish
    with torch.inference_mode():
        out = model.generate(
            **inp,
            max_new_tokens=100,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id
            if tok.pad_token_id is not None
            else tok.eos_token_id
        )

    # Faqat yangi yaratilgan tokenlar
    input_length = inp["input_ids"].shape[-1]
    new_tokens = out[0][input_length:]

    return tok.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


# Barcha savollarga javob olish
natijalar = [
    {"savol": s, "instruct_javob_2": javob(s)}
    for s in SAVOLLAR
]

# Natijalarni chiqarish
for r in natijalar:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["instruct_javob_2"])
    print("-" * 70)

# JSON faylga saqlash
with open(
    "instruct_model_chat_template_1dars.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        natijalar,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saqlandi: instruct_model_chat_template_1dars.json")


SAVOL: Google nima?
JAVOB: I'm sorry, but I can't assist with that.
----------------------------------------------------------------------
SAVOL: Italiya poytaxti qayer?
JAVOB: I apologize, but I'm not able to understand or respond to the phrase you provided. It doesn't appear to be in a language I recognize or can interpret. Could you please clarify what language this is in, or provide more context about what you're asking? If it's related to something specific (like a question, a topic, etc.), could you rephrase your request in that language? I'd be happy to try and help if you can give me more information about what you're looking
----------------------------------------------------------------------
SAVOL: Menga AIni organish uchun 3 ta maslahat ber.
JAVOB: Menga Aini organizasyonu (Organizational Chart) o'ylashini tushunuvchi katta amalga qilishni yordamida:

1. Organizatsiya: Tug'ilgan organizatsiyalar va organizatsiyalarimizda ishlar va ishlarni olish.

2. Organizatsiya: Tug'ilg

In [18]:
natijalar = []
for i, s in enumerate(SAVOLLAR, 1):
    print(f"[{i}/5]...")
    natijalar.append({"savol": s, "instruct_javob_2": javob(s)})

[1/5]...
[2/5]...
[3/5]...
[4/5]...
[5/5]...
